# Pedalia · 3.2.2 · La caja de herramientas de Pedalia

Solución de referencia: el notebook que construye la práctica de la guía, ejecutado. Sirve para comparar con el nuestro.

[← Volver a la guía de la sesión](3.2.2%20-%20args%2C%20kwargs%2C%20lambdas%20y%20m%C3%B3dulos.md)

## El módulo pedalia.py

In [1]:
%%writefile pedalia.py
# pedalia.py
"""Funciones de Pedalia: la tarifa y la lectura del registro de viajes."""

import math


def bloques_extra(minutos):
    """Bloques de 30 minutos (o fracción) que pasan de los 30 primeros."""
    return max(0, math.ceil((minutos - 30) / 30))


def precio_viaje(minutos, usuario, bici="mecánica", *, suplemento=0.40,
                 base_ocasional=1.50, bloque_ocasional=1.00, bloque_abonado=0.50):
    """Precio de un viaje de Pedalia, en euros.

    Los precios de la tarifa se pasan por nombre y valen, por defecto, los
    vigentes. Sirven para simular otras tarifas:
    precio_viaje(64, "ocasional", "eléctrica", suplemento=0.60)
    """
    if minutos <= 0:
        raise ValueError(f"duración no válida: {minutos}")

    if usuario == "abonado":
        precio = bloques_extra(minutos) * bloque_abonado
    elif usuario == "ocasional":
        precio = base_ocasional + bloques_extra(minutos) * bloque_ocasional
    else:
        raise ValueError(f"tipo de usuario desconocido: {usuario!r}")

    if bici == "eléctrica":
        precio += suplemento
    return round(precio, 2)


def leer_viaje(linea):
    """Convierte una línea del registro ("id;origen;destino;minutos;usuario;bici")
    en un diccionario de viaje. Lanza ValueError si la línea no es válida."""
    id_viaje, origen, destino, minutos, usuario, bici = linea.split(";")
    viaje = {
        "id": int(id_viaje),
        "origen": origen.strip(),
        "destino": destino.strip(),
        "minutos": int(minutos),
        "usuario": usuario.strip().lower(),
        "bici": bici.strip().lower(),
    }
    precio_viaje(viaje["minutos"], viaje["usuario"], viaje["bici"])   # valida el viaje
    return viaje


def leer_registro(lineas):
    """Lee las líneas de un registro. Devuelve (válidos, rechazados), donde cada
    rechazado es una tupla (línea, motivo)."""
    validos, rechazados = [], []
    for linea in lineas:
        try:
            validos.append(leer_viaje(linea))
        except ValueError as error:
            rechazados.append((linea, str(error)))
    return validos, rechazados


def resumen_por(viajes, clave, **tarifa):
    """Viajes e ingresos de cada grupo: {grupo: {"viajes": n, "ingresos": euros}}.

    clave   función que recibe un viaje y devuelve su grupo
    tarifa  precios de la tarifa que se quieran cambiar (ver precio_viaje)
    """
    resumen = {}
    for viaje in viajes:
        grupo = clave(viaje)
        precio = precio_viaje(viaje["minutos"], viaje["usuario"], viaje["bici"], **tarifa)
        if grupo not in resumen:
            resumen[grupo] = {"viajes": 0, "ingresos": 0.0}
        resumen[grupo]["viajes"] += 1
        resumen[grupo]["ingresos"] = round(resumen[grupo]["ingresos"] + precio, 2)
    return resumen

Writing pedalia.py


## Importar y probar el módulo

In [2]:
import pedalia

assert pedalia.bloques_extra(30) == 0
assert pedalia.bloques_extra(31) == 1
assert pedalia.bloques_extra(60) == 1
assert pedalia.bloques_extra(61) == 2
assert pedalia.precio_viaje(30, "abonado") == 0
assert pedalia.precio_viaje(31, "abonado") == 0.50
assert pedalia.precio_viaje(64, "ocasional", "eléctrica") == 3.90
assert pedalia.precio_viaje(64, "ocasional", "eléctrica", suplemento=0.60) == 4.10
print("Todas las pruebas pasan")

Todas las pruebas pasan


## Los viajes del martes

In [3]:
viajes = [
    {"id": 1,  "origen": "Plaza Mayor",          "destino": "Campus Universitario", "minutos": 14, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 2,  "origen": "Estación Central",     "destino": "Plaza Mayor",          "minutos": 9,  "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 3,  "origen": "Campus Universitario", "destino": "Parque del Río",       "minutos": 38, "usuario": "ocasional", "bici": "mecánica"},
    {"id": 4,  "origen": "Parque del Río",       "destino": "Mercado Central",      "minutos": 22, "usuario": "ocasional", "bici": "eléctrica"},
    {"id": 5,  "origen": "Hospital General",     "destino": "Estación Central",     "minutos": 31, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 6,  "origen": "Mercado Central",      "destino": "Plaza Mayor",          "minutos": 6,  "usuario": "abonado",   "bici": "mecánica"},
    {"id": 7,  "origen": "Plaza Mayor",          "destino": "Parque del Río",       "minutos": 75, "usuario": "ocasional", "bici": "mecánica"},
    {"id": 8,  "origen": "Estación Central",     "destino": "Hospital General",     "minutos": 17, "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 9,  "origen": "Campus Universitario", "destino": "Plaza Mayor",          "minutos": 12, "usuario": "abonado",   "bici": "mecánica"},
    {"id": 10, "origen": "Plaza Mayor",          "destino": "Estación Central",     "minutos": 11, "usuario": "abonado",   "bici": "eléctrica"},
    {"id": 11, "origen": "Parque del Río",       "destino": "Parque del Río",       "minutos": 64, "usuario": "ocasional", "bici": "eléctrica"},
    {"id": 12, "origen": "Mercado Central",      "destino": "Campus Universitario", "minutos": 19, "usuario": "abonado",   "bici": "mecánica"},
]

registro_tarde = [
    "13;Plaza Mayor;Mercado Central;8;abonado;mecánica",
    "14;Campus Universitario;Plaza Mayor;26;abonado;eléctrica",
    "15;Parque del Río;Parque del Río;95;ocasional;mecánica",
    "16;Estación Central;Hospital General;doce;abonado;mecánica",
    "17;Mercado Central;Parque del Río;41;Ocasional ;eléctrica",
    "18;Hospital General;Estación Central;-3;abonado;mecánica",
    "19;Plaza Mayor;Campus Universitario;33;abonado;eléctrica",
    "20;Parque del Río;Plaza Mayor;57;turista;mecánica",
    "21;Campus Universitario;Mercado Central;15",
]

validos, rechazados = pedalia.leer_registro(registro_tarde)
jornada = viajes + validos
print(f"Viajes del martes: {len(jornada)} válidos, {len(rechazados)} rechazados")
pedalia.resumen_por(jornada, lambda v: v["usuario"])

Viajes del martes: 17 válidos, 4 rechazados


{'abonado': {'viajes': 11, 'ingresos': 3.0},
 'ocasional': {'viajes': 6, 'ingresos': 19.2}}

## Ordenar y agrupar

In [4]:
caros = sorted(jornada,
               key=lambda v: pedalia.precio_viaje(v["minutos"], v["usuario"], v["bici"]),
               reverse=True)

for v in caros[:5]:
    precio = pedalia.precio_viaje(v["minutos"], v["usuario"], v["bici"])
    print(f"{v['id']:>3}  {v['usuario']:<10}{v['bici']:<10}{v['minutos']:>3} min {precio:>5.2f} €")

 15  ocasional mecánica   95 min  4.50 €
 11  ocasional eléctrica  64 min  3.90 €
  7  ocasional mecánica   75 min  3.50 €
 17  ocasional eléctrica  41 min  2.90 €
  3  ocasional mecánica   38 min  2.50 €


In [5]:
print(pedalia.resumen_por(jornada, lambda v: v["bici"]))
print(pedalia.resumen_por(jornada, lambda v: "más de 30 min" if v["minutos"] > 30 else "hasta 30 min"))

{'mecánica': {'viajes': 9, 'ingresos': 11.0}, 'eléctrica': {'viajes': 8, 'ingresos': 11.2}}
{'hasta 30 min': {'viajes': 10, 'ingresos': 3.5}, 'más de 30 min': {'viajes': 7, 'ingresos': 18.7}}


## Escenarios de tarifa

In [6]:
escenarios = {
    "Tarifa actual":                 {},
    "Suplemento eléctrico a 0,60 €": {"suplemento": 0.60},
    "Bloque de ocasional a 1,50 €":  {"bloque_ocasional": 1.50},
    "Bloque de abonado a 1,00 €":    {"bloque_abonado": 1.00},
}

for nombre, cambios in escenarios.items():
    resumen = pedalia.resumen_por(jornada, lambda v: v["usuario"], **cambios)
    total = sum(grupo["ingresos"] for grupo in resumen.values())
    print(f"{nombre:<31}{total:>6.2f} € · abonados {resumen['abonado']['ingresos']:>5.2f} € · "
          f"ocasionales {resumen['ocasional']['ingresos']:>5.2f} €")

Tarifa actual                   22.20 € · abonados  3.00 € · ocasionales 19.20 €
Suplemento eléctrico a 0,60 €   23.80 € · abonados  4.00 € · ocasionales 19.80 €
Bloque de ocasional a 1,50 €    26.70 € · abonados  3.00 € · ocasionales 23.70 €
Bloque de abonado a 1,00 €      23.20 € · abonados  4.00 € · ocasionales 19.20 €


## Escenarios de tarifa

- La tarifa vive ahora en el módulo `pedalia.py`, con todos sus precios como
  parámetros. Da las mismas cifras que la 3.2.1: 22,20 € el martes.
- Los viajes de más de 30 minutos son el 41 % de la jornada, pero generan el
  84 % de los ingresos por viaje, y los cinco más caros son de ocasionales.
- Subir el bloque adicional de los ocasionales a 1,50 € es el cambio que más
  ingresa (+4,50 € al día, un 20 %), y no afecta a los abonados. Subir el
  suplemento a 0,60 € suma 1,60 €, de los que 1 € pagan los abonados.
- Son 17 viajes de un solo día. Antes de proponer nada hay que repetirlo con
  los viajes del año (parte 3.3) y estimar cuántos viajes largos se perderían
  con una tarifa más cara.